In [1]:
import pandas as pd
from pathlib import Path
import sqlite3

In [2]:
cwd = Path.cwd()
project_root = cwd.parent if cwd.name.lower() == "notebooks" else cwd
db_dir = project_root / "data" / "raw"
db_dir.mkdir(parents=True, exist_ok=True)

con = sqlite3.connect(db_dir / "stroke.db")

In [3]:
raw = pd.read_sql("SELECT * FROM stroke", con)
print(f"Loaded: {raw.shape[0]} rows, {raw.shape[1]} columns.")

Loaded: 6053 rows, 27 columns.


In [4]:
target = "stroke"
excluded = [target, "SEQN", "alcohol_frequency"]

candidate_features = [
    column for column in raw.columns
    if column not in excluded
]

print("Outcome counts:")
print(raw[target].value_counts(dropna=False))

print("\nCandidate feature types:")
print(raw[candidate_features].dtypes)

Outcome counts:
stroke
0    5773
1     280
Name: count, dtype: int64

Candidate feature types:
gender                          str
age                           int64
race                            str
weight_kg                   float64
height_cm                   float64
bmi                         float64
bmi_category                    str
education                       str
marital_status                  str
employment_status               str
diabetes                    float64
hypertension                float64
high_cholesterol            float64
congestive_heart_failure    float64
coronary_heart_disease      float64
angina                      float64
heart_attack                float64
cancer                      float64
thyroid_problem             float64
liver_condition             float64
kidney_disease              float64
sleep_duration_hours        float64
sleep_bedtime                   str
smoking_status                  str
dtype: object


create dataset with complete cases

In [5]:
target = "stroke"

numeric_features = [
    "age", "bmi", "sleep_duration_hours"
]

categorical_features = [
    "gender", "race", "education", "marital_status", "employment_status",
    "diabetes", "hypertension", "high_cholesterol",
    "congestive_heart_failure", "coronary_heart_disease", "angina",
    "heart_attack", "cancer", "thyroid_problem", "liver_condition",
    "kidney_disease", "sleep_bedtime", "smoking_status"
]

features = numeric_features + categorical_features

In [6]:
complete_mask = raw[features].notna().all(axis=1) & raw[target].notna()

print(f"Complete cases: {complete_mask.sum()} of {len(raw)}")
print("\nStroke counts among complete cases:")
print(raw.loc[complete_mask, target].value_counts())

Complete cases: 5744 of 6053

Stroke counts among complete cases:
stroke
0    5493
1     251
Name: count, dtype: int64


In [7]:
complete_case = raw.loc[complete_mask, features + [target]].copy()

train/test split

In [10]:
from sklearn.model_selection import train_test_split

X = raw[features].copy()
y = raw[target].copy()

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size = 0.2,
    random_state = 42,
    stratify = y
)

print("Train rows: ", len(X_train))
print("Test rows: ", len(X_test))
print("Train stroke rate: ", round(y_train.mean() * 100, 2), "%")
print("Test stroke rate: ", round(y_test.mean() * 100, 2), "%")

Train rows:  4842
Test rows:  1211
Train stroke rate:  4.63 %
Test stroke rate:  4.62 %


Imputation

In [11]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

In [13]:
X_train_model = X_train.copy()
X_test_model = X_test.copy()

for frame in (X_train_model, X_test_model):
    frame[categorical_features] = (
        frame[categorical_features]
        .astype("string")
        .fillna("Missing")
        .astype(object)
    )

numeric_preprocessor = SimpleImputer(
    strategy="median",
    add_indicator=True
)

categorical_preprocessor = OneHotEncoder(handle_unknown="ignore")

preprocessor = ColumnTransformer(transformers=[
    ("numeric", numeric_preprocessor, numeric_features),
    ("categorical", categorical_preprocessor, categorical_features)
])

X_train_ready = preprocessor.fit_transform(X_train_model)
X_test_ready = preprocessor.transform(X_test_model)

print("Prepared train shape:", X_train_ready.shape)
print("Prepared test shape:", X_test_ready.shape)

Prepared train shape: (4842, 129)
Prepared test shape: (1211, 129)


Establish a baseline model

In [14]:
from sklearn.dummy import DummyClassifier
from sklearn.metrics import classification_report, confusion_matrix 

baseline = DummyClassifier(strategy="most_frequent")
baseline.fit(X_train_ready, y_train)

baseline_predictions = baseline.predict(X_test_ready)

print("Confusion matrix:")
print(confusion_matrix(y_test, baseline_predictions))

print("\nClassification report:")
print(classification_report(
    y_test,
    baseline_predictions,
    zero_division=0
))

Confusion matrix:
[[1155    0]
 [  56    0]]

Classification report:
              precision    recall  f1-score   support

           0       0.95      1.00      0.98      1155
           1       0.00      0.00      0.00        56

    accuracy                           0.95      1211
   macro avg       0.48      0.50      0.49      1211
weighted avg       0.91      0.95      0.93      1211



randomforestclassifier

In [17]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    average_precision_score,
    classification_report,
    confusion_matrix
)
from sklearn.pipeline import Pipeline

stroke_model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", RandomForestClassifier(
        n_estimators=300,
        class_weight="balanced",
        random_state=42,
        n_jobs=-1
    ))
])

stroke_model.fit(X_train_model, y_train)

predictions = stroke_model.predict(X_test_model)
probabilities = stroke_model.predict_proba(X_test_model)[:, 1]

print("Confusion matrix:")
print(confusion_matrix(y_test, predictions, labels=None))

print("\nClassification report:")
print(classification_report(y_test, predictions, zero_division=0))

print("Average precision:", round(
    average_precision_score(y_test, probabilities), 3
))

Confusion matrix:
[[1148    7]
 [  55    1]]

Classification report:
              precision    recall  f1-score   support

           0       0.95      0.99      0.97      1155
           1       0.12      0.02      0.03        56

    accuracy                           0.95      1211
   macro avg       0.54      0.51      0.50      1211
weighted avg       0.92      0.95      0.93      1211

Average precision: 0.136


use the complete-case dataset

In [18]:
train_complete_mask = X_train.notna().all(axis=1)
test_complete_mask = X_test.notna().all(axis=1)

X_train_complete = X_train.loc[train_complete_mask].copy()
y_train_complete = y_train.loc[train_complete_mask]

X_test_complete = X_test.loc[test_complete_mask].copy()
y_test_complete = y_test.loc[test_complete_mask]

print("Complete training rows:", len(X_train_complete))
print("Training stroke counts:")
print(y_train_complete.value_counts())

print("\nComplete test rows:", len(X_test_complete))
print("Test stroke counts:")
print(y_test_complete.value_counts())

Complete training rows: 4592
Training stroke counts:
stroke
0    4390
1     202
Name: count, dtype: int64

Complete test rows: 1152
Test stroke counts:
stroke
0    1103
1      49
Name: count, dtype: int64


compare the two datasets

In [19]:
from sklearn.base import clone
from sklearn.metrics import precision_score, recall_score

X_train_complete_model = X_train_complete.copy()
X_test_complete_model = X_test_complete.copy()

for frame in (X_train_complete_model, X_test_complete_model):
    frame[categorical_features] = (
        frame[categorical_features]
        .astype("string")
        .fillna("Missing")
        .astype(object)
    )

complete_case_model = Pipeline(steps=[
    ("preprocessor", clone(preprocessor)),
    ("classifier", RandomForestClassifier(
        n_estimators=300,
        class_weight="balanced",
        random_state=42,
        n_jobs=-1
    ))
])

complete_case_model.fit(X_train_complete_model, y_train_complete)

for name, model in [
    ("imputed model", stroke_model),
    ("Complete-case model", complete_case_model)
]:
    pred = model.predict(X_test_complete_model)
    prob = model.predict_proba(X_test_complete_model)[:, 1]

    print(f"\n{name}")
    print(confusion_matrix(y_test_complete, pred, labels=[0,1]))
    print("Stroke precision: ", round(
        precision_score(y_test_complete, pred, zero_division=0), 3 
    ))

    print("Stroke recall: ", round(
        recall_score(y_test_complete, pred, zero_division=0), 3
    ))

    print("Average precision: ", round(
        average_precision_score(y_test_complete, prob), 3
    ))

    



imputed model
[[1097    6]
 [  48    1]]
Stroke precision:  0.143
Stroke recall:  0.02
Average precision:  0.129

Complete-case model
[[1096    7]
 [  48    1]]
Stroke precision:  0.125
Stroke recall:  0.02
Average precision:  0.116


changing the threshold

In [20]:
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.metrics import precision_score, recall_score, f1_score

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

oof_probabilities = cross_val_predict(
    stroke_model,
    X_train_model,
    y_train,
    cv=cv,
    method="predict_proba",
    n_jobs=1
)[:,1]

threshold_rows = []

for threshold in [0.05, 0.10, 0.15, 0.20, 0.30, 0.40, 0.50]:
    predicted = (oof_probabilities >= threshold).astype(int)

    threshold_rows.append({
        "threshold": threshold,
        "predicted_strokes": int(predicted.sum()),
        "precision": precision_score(y_train, predicted, zero_division=0),
        "recall": recall_score(y_train, predicted, zero_division=0),
        "f1": f1_score(y_train, predicted, zero_division=0)
    })

threshold_summary = pd.DataFrame(threshold_rows)
print(threshold_summary.round(3).to_string(index=False))


 threshold  predicted_strokes  precision  recall    f1
      0.05               2384      0.087   0.924 0.159
      0.10               1612      0.109   0.781 0.191
      0.15               1156      0.130   0.670 0.217
      0.20                823      0.148   0.545 0.233
      0.30                368      0.163   0.268 0.203
      0.40                151      0.132   0.089 0.107
      0.50                 47      0.106   0.022 0.037


i want to catch all stroke patients but not at the cost of alarming a lot of people for no reason. I chose a cutoff of 15% of the whole group.

In [21]:
import numpy as np

max_flagged_rate = 0.15
eligible_thresholds = []

for threshold in np.sort(np.unique(oof_probabilities)):
    oof_predictions = (oof_probabilities >= threshold).astype(int)
    flagged_rate = oof_predictions.mean()

    if flagged_rate <= max_flagged_rate:
        eligible_thresholds.append({
            "threshold": threshold,
            "flagged_pct": 100 * flagged_rate,
            "precision": precision_score(y_train, oof_predictions, zero_division=0),
            "recall": recall_score(y_train, oof_predictions, zero_division=0)
        })

chosen = max(
    eligible_thresholds,
    key=lambda row: (row["recall"], row["precision"])
)
chosen_threshold = chosen["threshold"]

print(f"Chosen threshold: {chosen_threshold:.4f}")
print(f"Training CV flagged: {chosen['flagged_pct']:.1f}%")
print(f"Training CV precision: {chosen['precision']:.3f}")
print(f"Training CV recall: {chosen['recall']:.3f}")

Chosen threshold: 0.2200
Training CV flagged: 14.8%
Training CV precision: 0.155
Training CV recall: 0.496


In [22]:
test_predictions_at_threshold = (
    probabilities >= chosen_threshold
).astype(int)

print("\nHeld-out test results:")
print(f"Flagged: {test_predictions_at_threshold.sum()} of {len(y_test)} "
      f"({100 * test_predictions_at_threshold.mean():.1f}%)")
print(confusion_matrix(y_test, test_predictions_at_threshold, labels=[0, 1]))
print("Test precision:", round(
    precision_score(y_test, test_predictions_at_threshold, zero_division=0), 3
))
print("Test recall:", round(
    recall_score(y_test, test_predictions_at_threshold, zero_division=0), 3
))


Held-out test results:
Flagged: 195 of 1211 (16.1%)
[[986 169]
 [ 30  26]]
Test precision: 0.133
Test recall: 0.464


pretty decent result catches half of all stroke victims. but i do want this to be better there are way too many people getting false alarms and not all strokes are caught.

I am going to try a logistic regression

In [23]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_validate
from sklearn.preprocessing import StandardScaler

logistic_preprocessor = ColumnTransformer(transformers=[
    ("numeric", Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="median", add_indicator=True)),
        ("scaler", StandardScaler())
    ]), numeric_features),
    ("categorical", OneHotEncoder(handle_unknown="ignore"), categorical_features)
])

logistic_model = Pipeline(steps=[
    ("preprocessor", logistic_preprocessor),
    ("classifier", LogisticRegression(
        class_weight="balanced",
        max_iter=2000
    ))
])

comparison between logistic and rf

In [24]:
models = {
    "Random Forest": stroke_model,
    "Logistic Regression": logistic_model
}

for name, model in models.items():
    scores = cross_validate(
        model,
        X_train_model,
        y_train,
        cv=cv,
        scoring="average_precision",
        n_jobs = 1
    )["test_score"]

    print(f"{name}: average precision = "
          f"{scores.mean():.3f} +/- {scores.std():.3f}")

Random Forest: average precision = 0.143 +/- 0.020
Logistic Regression: average precision = 0.159 +/- 0.028


In [25]:
from sklearn.model_selection import cross_val_predict

logistic_oof_probabilities = cross_val_predict(
    logistic_model,
    X_train_model,
    y_train,
    cv=cv,
    method="predict_proba",
    n_jobs=1
)[:,1]

In [26]:
eligible_logistic_thresholds = []

for threshold in np.sort(np.unique(logistic_oof_probabilities)):
    predictions_at_threshold = (
        logistic_oof_probabilities >= threshold
    ).astype(int)
    flagged_rate = predictions_at_threshold.mean()

    if flagged_rate <= 0.15:
        eligible_logistic_thresholds.append({
            "threshold": threshold,
            "flagged_pct": 100 * flagged_rate,
            "precision": precision_score(
                y_train, predictions_at_threshold, zero_division=0
            ),
            "recall": recall_score(
                y_train, predictions_at_threshold, zero_division=0
            )
        })

chosen_logistic = max(
    eligible_logistic_thresholds,
    key=lambda row: (row["recall"], row["precision"])
)

print(f"Chosen threshold: {chosen_logistic['threshold']:.4f}")
print(f"Training CV flagged: {chosen_logistic['flagged_pct']:.1f}%")
print(f"Training CV precision: {chosen_logistic['precision']:.3f}")
print(f"Training CV recall: {chosen_logistic['recall']:.3f}")

Chosen threshold: 0.6873
Training CV flagged: 14.8%
Training CV precision: 0.155
Training CV recall: 0.496


In [27]:
logistic_model.fit(X_train_model, y_train)

logistic_test_probabilities = logistic_model.predict_proba(X_test_model)[:, 1]
logistic_test_predictions = (
    logistic_test_probabilities >= chosen_logistic["threshold"]
).astype(int)

print("Flagged:", logistic_test_predictions.sum(), "of", len(y_test))
print(confusion_matrix(y_test, logistic_test_predictions, labels=[0, 1]))
print("Test precision:", round(
    precision_score(y_test, logistic_test_predictions, zero_division=0), 3
))
print("Test recall:", round(
    recall_score(y_test, logistic_test_predictions, zero_division=0), 3
))
print("Test average precision:", round(
    average_precision_score(y_test, logistic_test_probabilities), 3
))

Flagged: 197 of 1211
[[980 175]
 [ 34  22]]
Test precision: 0.112
Test recall: 0.393
Test average precision: 0.125
